In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv('../Datasets/Reviews.csv')

# Keep only users who have rated at least 10 products
# This reduces matrix size massively
user_counts = df['UserId'].value_counts()
df = df[df['UserId'].isin(user_counts[user_counts >= 10].index)]

cf_df = df[['UserId', 'ProductId', 'Score']]
print("Shape after filtering:", cf_df.shape)
print("Unique users:", cf_df['UserId'].nunique())
print("Unique products:", cf_df['ProductId'].nunique())

Shape after filtering: (141294, 3)
Unique users: 7590
Unique products: 27385


In [3]:
# Rows = Users, Columns = Products, Values = Scores
user_product_matrix = cf_df.pivot_table(
    index='UserId', 
    columns='ProductId', 
    values='Score'
).fillna(0)

print("Matrix shape:", user_product_matrix.shape)
user_product_matrix.head()

Matrix shape: (7590, 27385)


ProductId,7310172001,7310172101,7800648702,B00002N8SM,B00004CI84,B00004CXX9,B00004RAMV,B00004RAMX,B00004RAMY,B00004RBDU,...,B009M2LRTA,B009M2LUEW,B009M4JDWQ,B009NTCO4O,B009NY1MC4,B009PCDDO4,B009QEBGIQ,B009QNJRSS,B009RB4GO4,B009SA5NNW
UserId,,,,,,,,,,,,,,,,,,,,,
A100WO06OQR8BQ,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A101P2KHWCU0G6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A102LH0KD8SYHX,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A102TGNH1D915Z,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A102UXGLDF76G1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
# Calculate similarity between all users
user_similarity = cosine_similarity(user_product_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_product_matrix.index,
    columns=user_product_matrix.index
)
print("User similarity matrix created!")
user_similarity_df.head()

User similarity matrix created!


UserId,A100WO06OQR8BQ,A101P2KHWCU0G6,A102LH0KD8SYHX,A102TGNH1D915Z,A102UXGLDF76G1,A103U3KR4L2ZXT,A10443LYX7DN89,A1045NW0WUEBP8,A104NBZOSZODEQ,A1051DBTLWP5A2,...,AZOYVQTM4DWAX,AZQXSDY803256,AZRJH4JFB59VC,AZU4M5K1N2LAB,AZU8GQQW6HESP,AZUUU81LB0NYV,AZV26LP92E6WU,AZWRZZAMX90VT,AZXKAH2DE6C8A,AZYMD9P9F9UZ6
UserId,,,,,,,,,,,,,,,,,,,,,
A100WO06OQR8BQ,1.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A101P2KHWCU0G6,0.0,1.000000,0.838835,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A102LH0KD8SYHX,0.0,0.838835,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A102TGNH1D915Z,0.0,0.000000,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A102UXGLDF76G1,0.0,0.000000,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
def predict_ratings(user_product_matrix, user_similarity):
    # Weighted average of similar users ratings
    mean_user_rating = user_product_matrix.mean(axis=1)
    ratings_diff = user_product_matrix.sub(mean_user_rating, axis=0)
    predicted = mean_user_rating.values[:, np.newaxis] + (
        user_similarity.dot(ratings_diff) / 
        np.abs(user_similarity).sum(axis=1)[:, np.newaxis]
    )
    return pd.DataFrame(
        predicted,
        index=user_product_matrix.index,
        columns=user_product_matrix.columns
    )

predicted_ratings = predict_ratings(user_product_matrix, user_similarity)
print("Ratings predicted successfully!")

Ratings predicted successfully!


In [6]:
# Get actual vs predicted for rated products only
actual = user_product_matrix.values.flatten()
predicted = predicted_ratings.values.flatten()

# Only evaluate where actual ratings exist (non zero)
mask = actual != 0
actual_rated = actual[mask]
predicted_rated = predicted[mask]

mae = mean_absolute_error(actual_rated, predicted_rated)
mse = mean_squared_error(actual_rated, predicted_rated)
r2 = r2_score(actual_rated, predicted_rated)

print(f"MAE  : {round(mae, 4)}")
print(f"MSE  : {round(mse, 4)}")
print(f"R²   : {round(r2, 4)}")

MAE  : 2.5101
MSE  : 8.8886
R²   : -5.156


In [7]:
def recommend_products(user_id, num_recommendations=5):
    # Products already rated by user
    rated = user_product_matrix.loc[user_id]
    rated_products = rated[rated > 0].index.tolist()
    
    # Get predicted scores for unrated products
    user_predictions = predicted_ratings.loc[user_id]
    unrated = user_predictions.drop(rated_products)
    
    # Top recommendations
    top = unrated.nlargest(num_recommendations)
    
    print(f"Top {num_recommendations} recommendations for user:")
    for i, (product, score) in enumerate(top.items(), 1):
        print(f"{i}. Product: {product} | Predicted Score: {round(score, 2)}")

# Test with first user
sample_user = user_product_matrix.index[0]
recommend_products(sample_user)

Top 5 recommendations for user:
1. Product: B002IEVJRY | Predicted Score: 0.92
2. Product: B002IEZJMA | Predicted Score: 0.88
3. Product: B0051COPH6 | Predicted Score: 0.73
4. Product: B0041NYV8E | Predicted Score: 0.73
5. Product: B005HG9ERW | Predicted Score: 0.62


## Observations
- Low R2 score is due to data sparsity - most user rated very few products
- This is known limitation  of CF on sparsity datasets
- The hybrid model will address these limitation

# Summary
CF finds users similar to you, looks at what they liked, and recommends those products to you!